In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import math

In [ ]:
# ---------------------------------------------------------------------------
# 1. Dataset (10 short sentences)
# ---------------------------------------------------------------------------
SENTENCES = [
    "the cat sat on the mat",
    "the dog ran in the park",
    "natural language processing is fascinating",
    "transformers use self attention",
    "deep learning powers modern AI",
    "the quick brown fox jumps high",
    "language models learn from text",
    "attention is all you need",
    "neural networks learn representations",
    "self attention captures long range dependencies",
]

In [ ]:
# ---------------------------------------------------------------------------
# 2. Simple whitespace tokenizer + vocabulary
# ---------------------------------------------------------------------------
def build_vocab(sentences):
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for s in sentences:
        for w in s.lower().split():
            if w not in vocab:
                vocab[w] = len(vocab)
    return vocab

def tokenize(sentences, vocab, max_len=None):
    seqs = [[vocab.get(w, 1) for w in s.lower().split()] for s in sentences]
    if max_len is None:
        max_len = max(len(s) for s in seqs)
    padded  = [s + [0] * (max_len - len(s)) for s in seqs]
    lengths = [len(s) for s in seqs]
    return torch.tensor(padded, dtype=torch.long), lengths, max_len

In [ ]:
# ---------------------------------------------------------------------------
# 3. Sinusoidal Positional Encoding
# ---------------------------------------------------------------------------
class SinusoidalPE(nn.Module):
    """
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    Non-learned; allows generalization to unseen sequence lengths.
    """
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)   # even dimensions
        pe[:, 1::2] = torch.cos(pos * div)   # odd dimensions
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        """x: (B, T, D)"""
        return self.dropout(x + self.pe[:, :x.size(1), :])


In [ ]:
# ---------------------------------------------------------------------------
# 4. Scaled Dot-Product Attention
# ---------------------------------------------------------------------------
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) V
    Q,K: (..., T, d_k)   V: (..., T, d_v)
    Returns output (..., T, d_v) and weights (..., T, T)
    """
    d_k    = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask, float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    weights = torch.nan_to_num(weights, nan=0.0)   # handle all-masked rows
    return torch.matmul(weights, V), weights

In [ ]:
# ---------------------------------------------------------------------------
# 5. Multi-Head Attention
# ---------------------------------------------------------------------------
class MultiHeadAttention(nn.Module):
    """
    Projects Q,K,V into h subspaces of size d_k = d_model // h,
    runs attention in each head, then concatenates and re-projects.
    Stores last attention weights for visualization.
    """
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.h   = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.last_weights = None   # saved for heatmap

    def forward(self, x, mask=None):
        B, T, D = x.shape
        # Project and reshape to (B, h, T, d_k)
        Q = self.W_q(x).view(B, T, self.h, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, T, self.h, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, T, self.h, self.d_k).transpose(1, 2)

        out, w = scaled_dot_product_attention(Q, K, V, mask)
        self.last_weights = w.detach()   # (B, h, T, T)

        # Concat heads and final projection
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.W_o(out)

In [ ]:
# ---------------------------------------------------------------------------
# 6. Position-wise Feed-Forward Network
# ---------------------------------------------------------------------------
class FFN(nn.Module):
    """Two-layer FFN: d_model -> d_ff -> d_model  with ReLU."""
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

In [ ]:
# ---------------------------------------------------------------------------
# 7. Encoder Layer  (Sub-layer 1: MHA + Add&Norm, Sub-layer 2: FFN + Add&Norm)
# ---------------------------------------------------------------------------
class EncoderLayer(nn.Module):
    """
    Single Transformer encoder layer.
    Add & Norm = residual connection + LayerNorm.
    Benefits: (1) stable gradient flow via skip connection,
              (2) reduced internal covariate shift via LayerNorm.
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn  = MultiHeadAttention(d_model, num_heads)
        self.ffn   = FFN(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Sub-layer 1: multi-head self-attention + Add & Norm
        x = self.norm1(x + self.drop(self.attn(x, mask)))
        # Sub-layer 2: feed-forward + Add & Norm
        x = self.norm2(x + self.ffn(x))
        return x

In [ ]:
# ---------------------------------------------------------------------------
# 8. Full Mini Transformer Encoder
# ---------------------------------------------------------------------------
class MiniEncoder(nn.Module):
    def __init__(self, vocab_size, d_model=64, num_heads=4,
                 d_ff=128, num_layers=2, dropout=0.1):
        super().__init__()
        self.embed  = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pe     = SinusoidalPE(d_model, dropout=dropout)
        self.layers = nn.ModuleList(
            [EncoderLayer(d_model, num_heads, d_ff, dropout)
             for _ in range(num_layers)]
        )
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, padding_mask=None):
        """
        x: (B, T) token ids
        padding_mask: (B, T) True where PAD token
        """
        mask = None
        if padding_mask is not None:
            mask = padding_mask.unsqueeze(1).unsqueeze(2)  # (B,1,1,T)
        out = self.pe(self.embed(x))
        for layer in self.layers:
            out = layer(out, mask)
        return self.norm(out)

    @property
    def last_attn(self):
        """Return attention weights from the last encoder layer."""
        return self.layers[-1].attn.last_weights

In [ ]:
# ---------------------------------------------------------------------------
# 9. Attention Heatmap Visualization
# ---------------------------------------------------------------------------
def plot_attention_heatmap(weights, tokens, sent_idx=0, head_idx=0,
                            fname="q2_attention_heatmap.png"):
    """
    weights: (B, H, T, T)
    tokens : list of word strings for sentence sent_idx
    """
    w = weights[sent_idx, head_idx].numpy()
    n = len(tokens)
    w = w[:n, :n]

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(w, cmap="Blues", vmin=0, vmax=w.max())
    ax.set_xticks(range(n)); ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=11)
    ax.set_yticks(range(n)); ax.set_yticklabels(tokens, fontsize=11)
    ax.set_xlabel("Key (attended-to token)", fontsize=12)
    ax.set_ylabel("Query (attending token)", fontsize=12)
    ax.set_title(f"Self-Attention Heatmap\n\"{' '.join(tokens)}\"\n"
                 f"Layer 2, Head {head_idx+1}", fontsize=12)
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(fname, dpi=120)
    plt.close()
    print(f"Heatmap saved -> {fname}")


In [ ]:
# ---------------------------------------------------------------------------
# 10. Main
# ---------------------------------------------------------------------------
def main():
    torch.manual_seed(42)

    # Step 1 & 2: Tokenize
    vocab         = build_vocab(SENTENCES)
    vocab_size    = len(vocab)
    tokens, lens, max_len = tokenize(SENTENCES, vocab)
    padding_mask  = (tokens == 0)   # True at PAD positions

    print("=" * 60)
    print("Mini Transformer Encoder")
    print("=" * 60)
    print(f"Vocabulary size : {vocab_size}")
    print(f"Sentences       : {len(SENTENCES)}")
    print(f"Max seq length  : {max_len}")
    print(f"Token ids shape : {tokens.shape}  (batch, seq_len)\n")

    # Show input tokens per sentence
    idx2word = {v: k for k, v in vocab.items()}
    print("Input tokens:")
    for i, (s, length) in enumerate(zip(SENTENCES, lens)):
        toks = s.lower().split()
        ids  = tokens[i, :length].tolist()
        print(f"  [{i}] {toks}")
        print(f"       ids={ids}")

    # Step 3-4: Build model and run forward pass
    model = MiniEncoder(
        vocab_size=vocab_size,
        d_model=64,
        num_heads=4,     # 4-head attention as required
        d_ff=128,
        num_layers=2,
        dropout=0.0,     # disable for deterministic demo
    )
    model.eval()

    with torch.no_grad():
        embeddings = model(tokens, padding_mask)   # (B, T, d_model)

    # Step 5a: Final contextual embeddings
    print(f"\nFinal contextual embeddings shape: {embeddings.shape}")
    print("  = (batch_size=10, seq_len, d_model=64)")
    print(f"\nSample: embedding of '{SENTENCES[0].split()[0]}' (token 0, sentence 0):")
    print(" ", embeddings[0, 0, :8].numpy().round(4), "... (first 8 of 64 dims)")

    # Step 5b: Attention heatmap (sentence 3: "transformers use self attention")
    viz_idx    = 3
    viz_tokens = SENTENCES[viz_idx].lower().split()
    plot_attention_heatmap(
        model.last_attn, viz_tokens,
        sent_idx=viz_idx, head_idx=0,
        fname="q2_attention_heatmap.png"
    )

    # Print attention weight matrix
    w = model.last_attn[viz_idx, 0, :len(viz_tokens), :len(viz_tokens)].numpy()
    print(f"\nAttention weights for sentence [{viz_idx}]"
          f" = \"{SENTENCES[viz_idx]}\"")
    print(f"Layer 2, Head 1  ({len(viz_tokens)}x{len(viz_tokens)} matrix):\n")
    header = "".join(f"{t:>14}" for t in viz_tokens)
    print(f"{'':>14}{header}")
    for row_tok, row in zip(viz_tokens, w):
        vals = "".join(f"{v:>14.4f}" for v in row)
        print(f"{row_tok:>14}{vals}")

    print("\nDone.")


if __name__ == "__main__":
    main()

Mini Transformer Encoder
Vocabulary size : 44
Sentences       : 10
Max seq length  : 6
Token ids shape : torch.Size([10, 6])  (batch, seq_len)

Input tokens:
  [0] ['the', 'cat', 'sat', 'on', 'the', 'mat']
       ids=[2, 3, 4, 5, 2, 6]
  [1] ['the', 'dog', 'ran', 'in', 'the', 'park']
       ids=[2, 7, 8, 9, 2, 10]
  [2] ['natural', 'language', 'processing', 'is', 'fascinating']
       ids=[11, 12, 13, 14, 15]
  [3] ['transformers', 'use', 'self', 'attention']
       ids=[16, 17, 18, 19]
  [4] ['deep', 'learning', 'powers', 'modern', 'ai']
       ids=[20, 21, 22, 23, 24]
  [5] ['the', 'quick', 'brown', 'fox', 'jumps', 'high']
       ids=[2, 25, 26, 27, 28, 29]
  [6] ['language', 'models', 'learn', 'from', 'text']
       ids=[12, 30, 31, 32, 33]
  [7] ['attention', 'is', 'all', 'you', 'need']
       ids=[19, 14, 34, 35, 36]
  [8] ['neural', 'networks', 'learn', 'representations']
       ids=[37, 38, 31, 39]
  [9] ['self', 'attention', 'captures', 'long', 'range', 'dependencies']
       i